**Header**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BASE    = '/content/drive/MyDrive/churn_project/'
DATA    = BASE + 'data/raw/'
PROC    = BASE + 'data/processed/'
RESULTS = BASE + 'results/'

print("✓ Ready")

Mounted at /content/drive
✓ Ready


**Reload all 3 datasets:**

In [2]:
import zipfile

# Telecom
telecom_train = pd.read_csv(DATA + 'churn-bigml-80.csv')
telecom_test  = pd.read_csv(DATA + 'churn-bigml-20.csv')
telecom       = pd.concat([telecom_train, telecom_test], ignore_index=True)

# ISP
isp = pd.read_csv(DATA + 'internet_service_churn.csv')

# Insurance
with zipfile.ZipFile(DATA + 'Train.csv.zip', 'r') as z:
    z.extractall(DATA)
with zipfile.ZipFile(DATA + 'Test.csv.zip', 'r') as z:
    z.extractall(DATA)
ins_train = pd.read_csv(DATA + 'Train.csv')
ins_test  = pd.read_csv(DATA + 'Test.csv')
insurance = pd.concat([ins_train, ins_test], ignore_index=True)

# Churn column map
churn_cols = {
    'Insurance' : 'labels',
    'ISP'       : 'churn',
    'Telecom'   : 'Churn'
}

datasets = {
    'Insurance' : insurance,
    'ISP'       : isp,
    'Telecom'   : telecom
}

for name, df in datasets.items():
    print(f"{name:12} → {df.shape}")

print("✓ All datasets loaded")

Insurance    → (45211, 17)
ISP          → (72274, 11)
Telecom      → (3333, 20)
✓ All datasets loaded


**Step1:Handle missing values**

In [3]:
# Insurance has NaN in labels column — drop those rows
# Also check for missing values in features

cleaned = {}

for name, df in datasets.items():
    churn_col = churn_cols[name]
    before    = len(df)

    # Drop rows where churn label is missing (Insurance has these)
    df = df.dropna(subset=[churn_col])

    # For numeric columns — fill missing with median
    num_cols = df.select_dtypes(include='number').columns.tolist()
    num_cols = [c for c in num_cols if c != churn_col]
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    # For object columns — fill missing with mode
    obj_cols = df.select_dtypes(include='object').columns.tolist()
    for col in obj_cols:
        df[col] = df[col].fillna(df[col].mode()[0])

    after = len(df)
    cleaned[name] = df
    print(f"{name:12} → rows before: {before:,}  after: {after:,}  "
          f"dropped: {before - after}")

print("\n✓ Missing values handled")

Insurance    → rows before: 45,211  after: 33,908  dropped: 11303
ISP          → rows before: 72,274  after: 72,274  dropped: 0
Telecom      → rows before: 3,333  after: 3,333  dropped: 0

✓ Missing values handled


**Step2:Standardize churn column to 0/1 integer**

In [4]:
# Convert all churn columns to clean 0/1 integers
# Insurance: float 0.0/1.0 → int
# ISP: already int 0/1 — no change needed
# Telecom: bool False/True → int

for name, df in cleaned.items():
    churn_col = churn_cols[name]

    if df[churn_col].dtype == bool:
        df[churn_col] = df[churn_col].astype(int)
    elif df[churn_col].dtype == float:
        df[churn_col] = df[churn_col].astype(int)
    # int stays as is

    print(f"{name:12} → churn dtype: {df[churn_col].dtype}  "
          f"unique: {sorted(df[churn_col].unique())}")

print("\n✓ Churn columns standardized to 0/1")

Insurance    → churn dtype: int64  unique: [np.int64(0), np.int64(1)]
ISP          → churn dtype: int64  unique: [np.int64(0), np.int64(1)]
Telecom      → churn dtype: int64  unique: [np.int64(0), np.int64(1)]

✓ Churn columns standardized to 0/1


**Step3: Encode categorical columns:**

In [5]:
from sklearn.preprocessing import LabelEncoder

encoded = {}

for name, df in cleaned.items():
    churn_col = churn_cols[name]
    df        = df.copy()
    obj_cols  = df.select_dtypes(include='object').columns.tolist()

    print(f"\n{name} — encoding {len(obj_cols)} object columns:")

    for col in obj_cols:
        le        = LabelEncoder()
        df[col]   = le.fit_transform(df[col].astype(str))
        print(f"  '{col}' → {le.classes_[:4]}... → 0 to {len(le.classes_)-1}")

    encoded[name] = df

print("\n✓ Categorical columns encoded")


Insurance — encoding 0 object columns:

ISP — encoding 0 object columns:

Telecom — encoding 3 object columns:
  'State' → ['AK' 'AL' 'AR' 'AZ']... → 0 to 50
  'International plan' → ['No' 'Yes']... → 0 to 1
  'Voice mail plan' → ['No' 'Yes']... → 0 to 1

✓ Categorical columns encoded


**Step 4: Separate features and target**

In [6]:
X_dict = {}  # features
y_dict = {}  # labels

for name, df in encoded.items():
    churn_col    = churn_cols[name]
    X_dict[name] = df.drop(columns=[churn_col])
    y_dict[name] = df[churn_col]

    print(f"{name:12} → X: {X_dict[name].shape}  "
          f"y: {y_dict[name].shape}  "
          f"churn rate: {y_dict[name].mean()*100:.1f}%")

print("\n✓ Features and targets separated")

Insurance    → X: (33908, 16)  y: (33908,)  churn rate: 11.7%
ISP          → X: (72274, 10)  y: (72274,)  churn rate: 55.4%
Telecom      → X: (3333, 19)  y: (3333,)  churn rate: 14.5%

✓ Features and targets separated


**Step5: Train/Test split (80/20)**

In [7]:
from sklearn.model_selection import train_test_split

splits = {}  # will store X_train, X_test, y_train, y_test per dataset

for name in datasets.keys():
    X = X_dict[name]
    y = y_dict[name]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size    = 0.2,
        random_state = 42,
        stratify     = y   # keeps churn ratio same in train and test
    )

    splits[name] = {
        'X_train' : X_train,
        'X_test'  : X_test,
        'y_train' : y_train,
        'y_test'  : y_test
    }

    print(f"{name:12} → train: {X_train.shape}  "
          f"test: {X_test.shape}  "
          f"train churn: {y_train.mean()*100:.1f}%")

print("\n✓ Train/test split done (80/20, stratified)")

Insurance    → train: (27126, 16)  test: (6782, 16)  train churn: 11.7%
ISP          → train: (57819, 10)  test: (14455, 10)  train churn: 55.4%
Telecom      → train: (2666, 19)  test: (667, 19)  train churn: 14.5%

✓ Train/test split done (80/20, stratified)


**Step 6: Scale numeric features:**

In [8]:
from sklearn.preprocessing import StandardScaler

scalers = {}  # save scalers for later use in model notebooks

for name in datasets.keys():
    X_train = splits[name]['X_train'].copy()
    X_test  = splits[name]['X_test'].copy()

    scaler     = StandardScaler()
    # IMPORTANT: fit on train only, transform both
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    # Convert back to DataFrame to keep column names
    splits[name]['X_train_sc'] = pd.DataFrame(
        X_train_sc, columns=X_train.columns)
    splits[name]['X_test_sc']  = pd.DataFrame(
        X_test_sc,  columns=X_train.columns)

    scalers[name] = scaler
    print(f"{name:12} → scaled  mean≈{X_train_sc.mean():.4f}  "
          f"std≈{X_train_sc.std():.4f}")

print("\n✓ Features scaled (StandardScaler fit on train only)")

Insurance    → scaled  mean≈0.0000  std≈1.0000
ISP          → scaled  mean≈-0.0000  std≈1.0000
Telecom      → scaled  mean≈-0.0000  std≈1.0000

✓ Features scaled (StandardScaler fit on train only)


**Step 7: Apply SMOTE(afetr split-correct way)**

In [9]:
from imblearn.over_sampling import SMOTE

smoted = {}  # balanced training data

for name in datasets.keys():
    X_train = splits[name]['X_train_sc']
    y_train = splits[name]['y_train']

    before_0 = (y_train == 0).sum()
    before_1 = (y_train == 1).sum()

    smote              = SMOTE(random_state=42)
    X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

    after_0 = (y_train_sm == 0).sum()
    after_1 = (y_train_sm == 1).sum()

    smoted[name] = {
        'X_train' : X_train_sm,
        'y_train' : y_train_sm,
        'X_test'  : splits[name]['X_test_sc'],
        'y_test'  : splits[name]['y_test']
    }

    print(f"{name}")
    print(f"  Before SMOTE → 0: {before_0:,}  1: {before_1:,}")
    print(f"  After  SMOTE → 0: {after_0:,}  1: {after_1:,}")
    print(f"  Churn rate after SMOTE: "
          f"{y_train_sm.mean()*100:.1f}%\n")

print("✓ SMOTE applied on training data only")

Insurance
  Before SMOTE → 0: 23,952  1: 3,174
  After  SMOTE → 0: 23,952  1: 23,952
  Churn rate after SMOTE: 50.0%

ISP
  Before SMOTE → 0: 25,779  1: 32,040
  After  SMOTE → 0: 32,040  1: 32,040
  Churn rate after SMOTE: 50.0%

Telecom
  Before SMOTE → 0: 2,280  1: 386
  After  SMOTE → 0: 2,280  1: 2,280
  Churn rate after SMOTE: 50.0%

✓ SMOTE applied on training data only


**Step 8: Save all processed data to Drive**

In [10]:
import pickle

# Save as pickle — preserves numpy arrays cleanly for model notebooks
pickle_path = PROC + 'smoted_data.pkl'
with open(pickle_path, 'wb') as f:
    pickle.dump(smoted, f)
print(f"✓ Smoted data saved → {pickle_path}")

# Also save scalers
scaler_path = PROC + 'scalers.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scalers, f)
print(f"✓ Scalers saved     → {scaler_path}")

# Save a readable summary
summary = []
for name in datasets.keys():
    summary.append({
        'Dataset'         : name,
        'X_train shape'   : str(smoted[name]['X_train'].shape),
        'X_test shape'    : str(smoted[name]['X_test'].shape),
        'Train churn rate': f"{smoted[name]['y_train'].mean()*100:.1f}%",
        'Test churn rate' : f"{smoted[name]['y_test'].mean()*100:.1f}%",
        'Features'        : smoted[name]['X_train'].shape[1]
    })

summary_df = pd.DataFrame(summary)
display(summary_df)
summary_df.to_csv(RESULTS + 'preprocessing_summary.csv', index=False)
print("\n✓ Preprocessing complete — all data saved to Drive")
print("✓ Next step: run 03_xgboost notebook")

✓ Smoted data saved → /content/drive/MyDrive/churn_project/data/processed/smoted_data.pkl
✓ Scalers saved     → /content/drive/MyDrive/churn_project/data/processed/scalers.pkl


,Dataset,X_train shape,X_test shape,Train churn rate,Test churn rate,Features
0,Insurance,"(47904, 16)","(6782, 16)",50.0%,11.7%,16
1,ISP,"(64080, 10)","(14455, 10)",50.0%,55.4%,10
2,Telecom,"(4560, 19)","(667, 19)",50.0%,14.5%,19



✓ Preprocessing complete — all data saved to Drive
✓ Next step: run 03_xgboost notebook
